In [55]:
import os
import pandas as pd
import geopandas as gpd
import re

In [3]:
# folder location of downloaded zip files
HI_dir = '/Users/jphuong/Downloads/HI_ADDR'
os.listdir(HI_dir)

['tl_2025_15001_addr.zip',
 'tl_2025_15007_addr.zip',
 'tl_2025_15009_addr.zip',
 'tl_2025_15003_addr.zip',
 'tl_2025_15005_addr.zip']

In [8]:
import zipfile

def unzip_file_path (zip_path, root_dir):
    """
    function to unzip all the zip files

    zip_path (dir): path to the zip file

    return: unzipped folder by the same name with the contents within
    """
    # start
    print('starting: {0}'.format(zip_path))

    # extract
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(os.path.join(root_dir, zip_path.replace('.zip','')))

    # stop
    print('done: {0}'.format(zip_path))

In [9]:
# unzip all zip folders
[unzip_file_path (zip_path=os.path.join(HI_dir, x), 
                  root_dir=HI_dir) for x in os.listdir(HI_dir)]

starting: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15001_addr.zip
done: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15001_addr.zip
starting: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15007_addr.zip
done: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15007_addr.zip
starting: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15009_addr.zip
done: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15009_addr.zip
starting: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15003_addr.zip
done: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15003_addr.zip
starting: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15005_addr.zip
done: /Users/jphuong/Downloads/HI_ADDR/tl_2025_15005_addr.zip


[None, None, None, None, None]

In [10]:
# zip files in root
zipf = [f for f in os.listdir(HI_dir) if f.endswith('.zip')]

# folders in root
fold = [f for f in os.listdir(HI_dir) if not f.endswith('.zip') and ('.' not in f)]

fold

['tl_2025_15007_addr',
 'tl_2025_15001_addr',
 'tl_2025_15003_addr',
 'tl_2025_15009_addr',
 'tl_2025_15005_addr']

In [20]:
# change working directory
os.chdir(os.path.join(HI_dir, fold[0]))

In [21]:
# inspect files in the unzipped folder
os.listdir()

['tl_2025_15007_addr.dbf.iso.xml',
 'tl_2025_15007_addr.dbf.ea.iso.xml',
 'tl_2025_15007_addr.dbf',
 'tl_2025_15007_addr.cpg']

In [22]:
# identify the .dbf file
[x for x in os.listdir() if x.endswith(".dbf")]

['tl_2025_15007_addr.dbf']

In [23]:
# map the path for the .dbf file
dbf_file_path = os.path.join(HI_dir, 
                             fold[0], 
                             [x for x in os.listdir() if x.endswith(".dbf")][0])

# read the dbf file
gdf = gpd.read_file(dbf_file_path)
gdf

,TLID,FROMHN,TOHN,SIDE,ZIP,PLUS4,FROMTYP,TOTYP,ARID,MTFCC
0,203387920,173,171,L,96741,None,None,I,40050176169,D1000
1,203387920,199,175,L,96705,None,None,None,40039465114217,D1000
2,203388338,499,401,R,96705,None,None,None,4005599331356,D1000
3,203389169,380,350,L,96705,None,None,I,4003978163172,D1000
4,614443320,2-4001,2-4033,L,96756,None,None,None,40039465059711,D1000
...,...,...,...,...,...,...,...,...,...,...
5775,614440774,4500,4598,R,96703,None,None,None,4003977969368,D1000
5776,203409658,4001,4099,L,96703,None,None,None,40019665491291,D1000
5777,635269023,4598,4500,L,96703,None,I,None,4003978172079,D1000
5778,655263893,4384,4300,L,96703,None,None,None,4003978129420,D1000


In [24]:
# search for TLID
gdf.query("TLID==203387920")

,TLID,FROMHN,TOHN,SIDE,ZIP,PLUS4,FROMTYP,TOTYP,ARID,MTFCC
0,203387920,173,171,L,96741,None,None,I,40050176169,D1000
1,203387920,199,175,L,96705,None,None,None,40039465114217,D1000
3639,203387920,149,145,L,96705,None,None,None,40039465114207,D1000
5052,203387920,169,167,L,96705,None,None,None,40039465114211,D1000


In [50]:
# search for TLID containing "-"
obj1 = gdf.query("FROMHN.str.contains('-')", engine='python')

# inspect character length
obj1['FROMHN_len'] = obj1.FROMHN.apply(lambda x: len(x))
obj1['FROMHN_len0'] = obj1.FROMHN.apply(lambda x: len(x.split('-')[0]))
obj1['FROMHN_len1'] = obj1.FROMHN.apply(lambda x: len(x.split('-')[1]))

obj1.groupby(['FROMHN_len','FROMHN_len0'])['TLID'].count()

#obj1.loc[obj1['FROMHN_len']==5,:].sort_values('TLID')

#obj1['FROMHN_len'].agg(['min','max'])

/var/folders/jh/4y0fvt6d0qq8lbbrshtzn41c0000gp/T/ipykernel_86417/1751652475.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obj1['FROMHN_len'] = obj1.FROMHN.apply(lambda x: len(x))
/var/folders/jh/4y0fvt6d0qq8lbbrshtzn41c0000gp/T/ipykernel_86417/1751652475.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obj1['FROMHN_len0'] = obj1.FROMHN.apply(lambda x: len(x.split('-')[0]))
/var/folders/jh/4y0fvt6d0qq8lbbrshtzn41c0000gp/T/ipykernel_86417/1751652475.py:7: SettingWithCopyWarning: 
A value is trying to

FROMHN_len  FROMHN_len0
5           1               32
6           1              273
            3                1
            4                8
7           2                1
            4               12
Name: TLID, dtype: int64

In [28]:
# search for TLID 614443320
gdf.query("TLID==614443320")

,TLID,FROMHN,TOHN,SIDE,ZIP,PLUS4,FROMTYP,TOTYP,ARID,MTFCC
4,614443320,2-4001,2-4033,L,96756,None,None,None,40039465059711,D1000
691,614443320,2-4000,2-4098,R,96756,None,None,None,4003977840841,D1000
692,614443320,2-4037,2-4099,L,96756,None,None,None,40039465059728,D1000


In [60]:
# find all TLIDs without "-" in the range
obj2 = gdf.query("~FROMHN.str.contains('-')", engine='python')

# inspect character length
obj2['FROMHN_len'] = obj2.FROMHN.apply(lambda x: len(x))
#obj2['FROMHN_len0'] = obj2.FROMHN.apply(lambda x: len(x.split('[a-zA-Z]')[0]) if re.search('[a-zA-Z]', x) else 0)
#obj2['FROMHN_len1'] = obj2.FROMHN.apply(lambda x: len(x.split('[a-zA-Z]')[1]) if re.search('[a-zA-Z]', x) else 0)

# identify long terms
obj2.loc[obj2['FROMHN_len']==7,:].sort_values('TLID')

#obj2.FROMHN.apply(lambda x: len(x)).agg(['min','max'])

/var/folders/jh/4y0fvt6d0qq8lbbrshtzn41c0000gp/T/ipykernel_86417/3935848827.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obj2['FROMHN_len'] = obj2.FROMHN.apply(lambda x: len(x))


,TLID,FROMHN,TOHN,SIDE,ZIP,PLUS4,FROMTYP,TOTYP,ARID,MTFCC,FROMHN_len
2313,203393867,2883B01,2883B99,L,96754,None,None,None,4003978051376,D1000,7
534,203398512,4330A99,4330A01,R,96766,None,None,None,4005599381696,D1000,7
4905,203398512,4330F98,4330F00,L,96766,None,None,None,4004702360595,D1000,7
4904,203398512,4330H98,4330H00,L,96766,None,None,None,4004702360580,D1000,7
4903,203398512,4430D98,4430D00,L,96766,None,None,None,4004702360648,D1000,7
4198,203398512,4330E98,4330E00,L,96766,None,None,None,4004702360603,D1000,7
4197,203398512,4330H99,4330H01,R,96766,None,None,None,4005599381706,D1000,7
4196,203398512,4330B99,4330B01,R,96766,None,None,None,4005599381700,D1000,7
3451,203398512,4330F99,4330F01,R,96766,None,None,None,4005599381731,D1000,7
2714,203398512,4330D98,4330D00,L,96766,None,None,None,4004702360610,D1000,7


In [61]:
# identify long terms
obj2.loc[obj2['FROMHN_len']==4,:].sort_values('TLID')

,TLID,FROMHN,TOHN,SIDE,ZIP,PLUS4,FROMTYP,TOTYP,ARID,MTFCC,FROMHN_len
2482,203385809,7900,7940,R,96752,None,None,None,40050179833,D1000,4
3976,203385872,8000,8098,R,96752,None,None,None,40050180153,D1000,4
1761,203385872,8007,8099,L,96752,None,None,None,40039465098778,D1000,4
2448,203385874,8098,8000,L,96752,None,None,None,40027724394047,D1000,4
3937,203385877,4598,4500,L,96752,None,None,None,4004032368286,D1000,4
...,...,...,...,...,...,...,...,...,...,...,...
5453,655264969,1898,1744,L,96756,None,None,None,4005599376035,D1000,4
373,655264969,1899,1753,R,96756,None,None,I,40050176860,D1000,4
2868,656320425,5800,5898,R,96746,None,None,None,40029552466683,D1000,4
2132,656320425,5893,5899,R,96746,None,None,None,40018383661244,D1000,4


In [62]:
obj2.groupby(['FROMHN_len'])['TLID'].count()

FROMHN_len
1      24
2      28
3     360
4    4999
5      17
6       5
7      20
Name: TLID, dtype: int64